# Experimentation Framework Demonstration
**Course**: AIE231 Neural Networks - Alamein International University  
**Role**: Experimentation Engineer  
**Deliverable**: Week 9 Experiment Tracking & Hyperparameter Tuning Infrastructure  

---
### Objectives:
1. Demonstrate the centralized experiment tracking protocol (`experiments/experiments.csv`).
2. Showcase the automated `ExperimentTrackingCallback` in Keras.
3. Demonstrate automated learning curve plotting with Overfitting / Underfitting detection.
4. Preview One-Factor-At-A-Time (OFAT) sweeps and multi-seed statistical aggregation.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.experiment_tracker import ExperimentTracker, ExperimentTrackingCallback
from src.plot_curves import plot_training_curves, plot_comparative_curves, plot_multi_seed_spread
from src.experiment_runner import build_mlp_model, load_data_or_mock

print("Modules imported successfully!")

## 1. Load Reference Configuration

In [ ]:
config_path = os.path.join(PROJECT_ROOT, "configs", "base_reference.json")
with open(config_path, "r") as f:
    reference_config = json.load(f)

print("Base Reference Configuration:")
print(json.dumps(reference_config, indent=2))

## 2. Inspect Existing Experiment Logs (`experiments.csv`)

In [ ]:
tracker = ExperimentTracker()
df_experiments = tracker.load_table()
print(f"Total logged runs: {len(df_experiments)}")
df_experiments.tail(5)

## 3. Run a Demonstration Training Run with Automatic Tracking

In [ ]:
import tensorflow as tf

# Load data (uses real dataset if present, or synthetic benchmark matching dimensions)
data_file = os.path.join(PROJECT_ROOT, "data", "bank-full.csv")
X_train, y_train, X_val, y_val, _, _ = load_data_or_mock(data_path=data_file if os.path.exists(data_file) else None)

# Build Model
demo_config = reference_config.copy()
demo_config["training"]["epochs"] = 15  # Fast demo run
model = build_mlp_model(input_dim=X_train.shape[1], config=demo_config)
model.summary()

# Set up tracking callback
run_id = f"EXP-DEMO-NB"
tracking_cb = ExperimentTrackingCallback(
    run_id=run_id,
    config=demo_config,
    tracker=tracker,
    plot_curves=True
)

# Train strictly on Train and Val splits!
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=demo_config["training"]["epochs"],
    batch_size=demo_config["training"]["batch_size"],
    callbacks=[tracking_cb],
    verbose=1
)

## 4. Visualize Generated Learning Curves

In [ ]:
save_curve_file = os.path.join(PROJECT_ROOT, "experiments", "curves", f"{run_id}_curves.png")
fig = plot_training_curves(
    history=tracking_cb.history,
    save_path=save_curve_file,
    title=f"Demo Learning Curves: {run_id}",
    primary_metric="pr_auc"
)
plt.show()

## 5. Verify the Updated Experiment Log

In [ ]:
updated_df = tracker.load_table()
print("Newly added entry:")
updated_df[updated_df["run_id"] == run_id][["run_id", "date", "model_name", "learning_rate", "batch_size", "val_loss", "val_pr_auc", "training_time_sec"]]